[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nilay47/marginally-realistic/blob/main/cvpr27/notebooks/nb4_eval.ipynb)

# nb4: evaluation (E1-E7)
Loads the **final EMA weights** of `joint_s0`, `video_only_s0`, `sensor_only_s0` and samples every variant (`joint`, `video_only`, `sensor_only`, `independent`, `cascaded`, `shuffle_oracle`): **3 draws x 1,024 samples, full 250-step ancestral**. Reports the main draw (draw 0) with 95% bootstrap CIs (B=200) and the spread (mean / std / min / max) across the 3 draws; then E3, E6 (sample efficiency), E7 (per-sample AUROC), the go/no-go flags (computed), and the figures. Sampling is resumable: each (variant, draw) .npz is written once and reused on rerun. Needs nb0-nb3 finished.

## 1. GPU

In [ ]:
import subprocess, torch
print(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout or "nvidia-smi: no output")
if not torch.cuda.is_available():
    raise SystemExit("No GPU: Runtime -> Change runtime type -> A100 GPU, then rerun.")
print("cuda:", torch.cuda.get_device_name(0), "| mem GB:", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))

## 2. Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os
ROOT = "/content/drive/MyDrive/physgen_cvpr27"
os.makedirs(ROOT, exist_ok=True)

## 3. Code

In [ ]:
import os
if os.path.exists("/content/repo"):
    !git -C /content/repo pull
else:
    !git clone https://github.com/nilay47/marginally-realistic.git /content/repo
%cd /content/repo
!git log -1 --format="commit %H%n%s"

## 4. Install

In [ ]:
# torch / torchvision come with Colab. pytorchvideo is the second I3D source for FVD (see videofeat.py).
!pip install -q scipy matplotlib pytorchvideo

## 5. Check inputs

In [ ]:
import os
for r in ["joint_s0", "video_only_s0", "sensor_only_s0"]:
    assert os.path.exists(f"{ROOT}/{r}/log.json"), f"{r} not finished (no log.json): run its training notebook first"
assert os.path.exists(f"{ROOT}/cache/video_metric.json"), "run nb0 first"
print("inputs present")

## 6. Sample + evaluate
Prints each sampling job's time. `--tag main` writes to `eval_main/` and refuses to overwrite: use a new `--tag` to rerun the metrics (cached samples are reused).

In [ ]:
%%bash
set -o pipefail
mkdir -p /content/drive/MyDrive/physgen_cvpr27/eval_main_logs
cd /content/repo
PYTHONUNBUFFERED=1 python -u -m cvpr27.eval --tag main 2>&1 | tee -a "/content/drive/MyDrive/physgen_cvpr27/eval_main_logs/eval_$(date +%s).log"

## 7. Figures (PDF + PNG)

In [ ]:
!cd /content/repo && python -u -m cvpr27.figures --tag main

## 8. Summary (read from results.json / results.md)

In [ ]:
from IPython.display import Markdown, Image, display
import glob, json
display(Markdown(open(f"{ROOT}/eval_main/results.md").read()))
for p in sorted(glob.glob(f"{ROOT}/eval_main/figures/*.png")):
    print(p); display(Image(p))